## Блок 1. `ndarray`, `shape`, `dtype` (задачи 1–4)

### Задача 1. Инвентаризация массива.
Создайте массив `np.arange(60).reshape(3, 4, 5)`. Выведите `shape`, `ndim`, `size`, `dtype`, `strides`, `C_CONTIGUOUS`. 

In [82]:
import numpy as np

a = np.arange(60).reshape(3, 4, 5)

print('shape: ', a.shape)
print('ndim: ', a.ndim)
print('size: ', a.size)
print('dtype: ', a.dtype)
print('strides: ', a.strides)
print('C_CONTIGUOUS: ', a.flags.c_contiguous)

shape:  (3, 4, 5)
ndim:  3
size:  60
dtype:  int64
strides:  (160, 40, 8)
C_CONTIGUOUS:  True


> что означает каждое число в `strides`:

Трехмерный массив можно представить как список двумерных таблиц (матриц). Т.е. у нас 3 таблицы по 4 строки и 5 колонок

Т.к. C_CONTIGUOUS:  True все элементы расположены в памяти друг за другом в едином непрерывном блоке памяти в построчном (C-style) порядке.
* 8 - размер одного элемента в байтах (соответствует dtype:  int64)
* 40 - размер строки в байтах = 5 * 8
* 160 - размер таблицы в байтах = 40 * 4

> почему `size = 60`

3 таблицы по 4 строки и 5 колонок это 3*4*5=60 чисел



### Задача 2. `int` vs `float` — тихая ловушка.
Создайте `a = np.array([1, 2, 3, 4])`. Выполните `a / 2`, `a // 2`, `a / 2.0`, `a.dtype`, `(a / 2).dtype`, `(a // 2).dtype`. Затем создайте `b = np.array([1, 2, 3], dtype=np.int8)`, прибавьте 127 и выведите результат и `dtype`.

In [83]:
a = np.array([1, 2, 3, 4])

print(f'a / 2         : {a / 2}')
print(f'a // 2        : {a // 2}')
print(f'a / 2.0       : {a / 2.0}')
print(f'a.dtype       : {a.dtype}')
print(f'(a / 2).dtype : {(a / 2).dtype}')
print(f'(a // 2).dtype: {(a // 2).dtype}')

b = np.array([1, 2, 3], dtype=np.int8)
print(f'b             : {b}')
print(f'b.dtype       : {b.dtype}')
b += 127
print(f'b + 127       : {b}')
print(f'(b+127).dtype : {b.dtype}')



a / 2         : [0.5 1.  1.5 2. ]
a // 2        : [0 1 1 2]
a / 2.0       : [0.5 1.  1.5 2. ]
a.dtype       : int64
(a / 2).dtype : float64
(a // 2).dtype: int64
b             : [1 2 3]
b.dtype       : int8
b + 127       : [-128 -127 -126]
(b+127).dtype : int8


Объясните: почему `a / 2` не меняет `a`, а `b + 127` даёт странный результат, и почему NumPy не падает с ошибкой.

> Ответ

* `a / 2` не меняет `a`, потому что арифметика NumPy по умолчанию создаёт **новый** массив для результата. Исходный `a` остаётся `int64`. Деление `/` всегда даёт `float64`, а `//` сохраняет целый тип, поэтому `(a / 2).dtype` — `float64`, а `(a // 2).dtype` — `int64`.
* С `b` другая ситуация: `b += 127` — это операция **на месте**, результат обязан остаться в том же массиве и в том же типе `int8` (диапазон от -128 до 127). Значения 128, 129, 130 не помещаются и «заворачиваются» по модулю 256 (как в дополнительном коде): 128 → -128, 129 → -127, 130 → -126. Без `+=` результат был бы такой же, потому что 127 тоже помещается в `int8`.
* NumPy не падает, потому что проверка переполнения на каждом элементе замедлила бы векторные вычисления. Целые типы фиксированной ширины работают как в C: переполнение происходит молча. Поэтому тип данных нужно выбирать осознанно.



### Задача 3. `float32` vs `float64` — реальная точность.
Сравните `np.float32(0.1) - np.float32(0.1)` не поможет; сделайте иначе: `float(np.float32(0.1)) - 0.1` и `float(np.float64(0.1)) - 0.1`. Выведите оба числа в `repr` и в `:.20e`. Посчитайте, во сколько раз ошибка `float32` больше ошибки `float64`.

> Я изменил метод расчета ошибки потому что 0.1 это и есть float64 в python и float(np.float64(0.1)) - 0.1 будет 0. repr(float) это тоже самое что и str(float)

> Я буду сравнивать с mpmath

In [84]:
from mpmath import mp

mp.dps = 50

delta1 = mp.mpf(float(np.float32(0.1))) - mp.mpf('0.1')
delta2 = mp.mpf(float(np.float64(0.1))) - mp.mpf('0.1')
print(f'Ошибка для float32: {mp.nstr(delta1, 10):>20} {repr(delta1)}')
print(f'Ошибка для float64: {mp.nstr(delta2, 10):>20} {repr(delta2)}')
print(f'delta1/delta2 = {mp.nstr(delta1/delta2, 3)}')


Ошибка для float32:       1.490116119e-9 mpf('0.0000000014901161193847656249999999999999999999999999331808822')
Ошибка для float64:      5.551115123e-18 mpf('5.5511151231257827021181583404541014956808822476951088e-18')
delta1/delta2 = 2.68e+8


Сделайте вывод: в каких задачах численных методов `float32` допустим, а в каких — нет.

> Вывод

Ошибка представления числа 0.1 в `float32` примерно 1.5·10⁻⁹ (относительная около 1.5·10⁻⁸), а в `float64` примерно 5.6·10⁻¹⁸. Это в ~2.7·10⁸ раз больше. Машинный эпсилон `float32` равен ≈1.2·10⁻⁷ (около 7 верных десятичных цифр), а `float64` ≈2.2·10⁻¹⁶ (около 15–16 цифр).

* `float32` допустим, когда сами данные неточны сильнее, чем 10⁻⁷ (измерения датчиков, изображения, звук), когда нужна скорость и память (большие массивы, GPU, обучение нейросетей), а вычисления короткие и без накопления ошибки.
* `float32` недопустим при длинных суммах и накоплении (миллионы слагаемых, интегрирование ОДУ на длинном интервале), при плохо обусловленных задачах (решение СЛАУ с большим числом обусловленности, где ошибка усиливается), при итерационных методах с жёстким допуском (точность 10⁻⁹ в `float32` недостижима), при вычитании близких чисел (катастрофическая потеря точности) и в финансовых расчётах.

### Задача 4. Явный `dtype` при создании.
Создайте четыре массива одинакового размера `(2, 3)`:

* `np.zeros` с `dtype=np.float64`;
* `np.zeros` с `dtype=np.float32`;
* `np.ones` с `dtype=np.int16`;
* `np.full` со значением `7` и `dtype=np.uint8`.

Выведите `dtype` и `nbytes` каждого. Посчитайте, сколько байт экономит `float32` по сравнению с `float64` на массиве `(1000, 1000)`.

In [85]:
z_f64 = np.zeros((2,3), dtype=np.float64)
z_f32 = np.zeros((2,3), dtype=np.float32)
o_i16 = np.ones((2,3), dtype=np.int16)
f = np.full((2,3), 7, dtype=np.uint8)

print('Размеры массивов 2x3 в байтах при разных типах данных')
print(f'Тип: {z_f64.dtype}; Размер всего массива в байтах {z_f64.nbytes}')
print(f'Тип: {z_f32.dtype}; Размер всего массива в байтах {z_f32.nbytes}')
print(f'Тип: {o_i16.dtype}; Размер всего массива в байтах {o_i16.nbytes}')
print(f'Тип: {f.dtype}; Размер всего массива в байтах {f.nbytes}')

e = (1000 * 1000)*(8 - 4)
print(f'Экономия float32 по сравнению с float64 на массиве 1000x1000 байт: {e:.1e}');


Размеры массивов 2x3 в байтах при разных типах данных
Тип: float64; Размер всего массива в байтах 48
Тип: float32; Размер всего массива в байтах 24
Тип: int16; Размер всего массива в байтах 12
Тип: uint8; Размер всего массива в байтах 6
Экономия float32 по сравнению с float64 на массиве 1000x1000 байт: 4.0e+06


 Сделайте вывод о компромиссе «память vs точность».

> Вывод

Каждый шаг вниз по типу уменьшает память в 2 раза: на массиве 1000×1000 `float64` занимает 8 МБ, `float32` — 4 МБ, то есть экономия 4·10⁶ байт (≈3.8 МиБ), и вдвое меньше данных пересылается в кэш и на GPU, поэтому вычисления быстрее. Платой становится точность (7 цифр вместо 15–16) и, для целых типов (`int16`, `uint8`), узкий диапазон значений (`uint8` только 0–255), что приводит к переполнению, как в задаче 2. Практическое правило: брать самый узкий тип, для которого точность и диапазон заведомо достаточны для задачи, а по умолчанию (когда есть сомнения) оставаться на `float64`.



## Блок 2. `reshape`, `ravel`, `flatten`, `.T`, `newaxis` (задачи 5–8)

### Задача 5. Цепочка `reshape`.
Создайте `a = np.arange(24)`. Преобразуйте последовательно в `(4, 6)`, `(2, 3, 4)`, `(2, 2, 6)`, `(24,)`. После каждого шага выведите `shape` и проверьте через `np.array_equal(a, текущий.ravel())`, что данные не потерялись.

In [86]:
a = np.arange(24)
a46 = a.reshape((4, 6))
print(f'shape: {a46.shape}; array is equal: {np.array_equal(a, a46.ravel())}; без копирования: {np.shares_memory(a, a46)}')
a234 = a46.reshape((2, 3, 4))
print(f'shape: {a234.shape}; array is equal: {np.array_equal(a, a234.ravel())}; без копирования: {np.shares_memory(a, a234)}')
a226 = a234.reshape((2, 2, 6))
print(f'shape: {a226.shape}; array is equal: {np.array_equal(a, a226.ravel())}; без копирования: {np.shares_memory(a, a226)}')
a24 = a226.reshape((24,))
print(f'shape: {a24.shape}; array is equal: {np.array_equal(a, a24.ravel())}; без копирования: {np.shares_memory(a, a24)}')


shape: (4, 6); array is equal: True; без копирования: True
shape: (2, 3, 4); array is equal: True; без копирования: True
shape: (2, 2, 6); array is equal: True; без копирования: True
shape: (24,); array is equal: True; без копирования: True


Объясните: почему `reshape` обычно не копирует данные и когда он **вынужден** скопировать.

> Ответ

Массив NumPy — это один плоский блок памяти плюс «описание»: `shape` и `strides`. `reshape` просто создаёт новое описание (новые `shape`/`strides`) над тем же блоком данных, поэтому он выполняется за O(1) и возвращает view (`np.shares_memory` дал `True` на каждом шаге).
Копия нужна, когда новую форму **нельзя** выразить через `strides` над существующей памятью, то есть когда элементы в памяти лежат не в том порядке, который требует новая форма. Типичный пример — не непрерывный массив, например транспонированный:

```python
x = np.arange(6).reshape(2, 3).T
print('x.reshape(6) разделяет память с x:', np.shares_memory(x, x.reshape(6)))
```

    x.reshape(6) разделяет память с x: False
    

Здесь `reshape` молча вернул копию, поэтому на view без проверки полагаться нельзя.


### Задача 6. `flatten` vs `ravel` — копия или view.

Создайте `M = np.arange(12).reshape(3, 4)`. Получите `f = M.flatten()` и `r = M.ravel()`. Выведите `f.base is None`, `r.base is M` (или `r.base is not None`). Измените `f[0] = 999` и `r[1] = 888` — проверьте, изменился ли `M` в каждом случае. 

In [87]:
m = np.arange(12).reshape(3, 4)
f = m.flatten()
r = m.ravel()
print(f'{f.base is None = }')
print(f'{r.base is None = }')
print(f'{r.base is m.base =}')
f[0] = 999
print(f'{f[0]=}; {m[0,0]=}')
r[1] = 888
print(f'{r[1]=}; {m[0,1]=}')




f.base is None = True
r.base is None = False
r.base is m.base =True
f[0]=np.int64(999); m[0,0]=np.int64(0)
r[1]=np.int64(888); m[0,1]=np.int64(888)


Сделайте вывод: когда использовать `ravel`, когда `flatten`.

> Вывод

`flatten()` всегда возвращает **копию** (`f.base is None`), поэтому `f[0] = 999` не затронул `M`. `ravel()` возвращает **view**, если массив непрерывен, поэтому `r[1] = 888` изменил `M[0, 1]`. Примечание: `r.base` совпадает с `m.base`, а не с `m`, потому что сам `m` — view на исходный `arange`, и NumPy хранит ссылку на изначального владельца данных.
Использовать `ravel`, когда нужно только прочитать данные или передать их дальше (быстрее, без расхода памяти). Использовать `flatten`, когда плоский массив собираются менять и исходный должен остаться нетронутым. На не непрерывном массиве `ravel` тоже скопирует данные, то есть заранее неочевидно, view это или копия, а `flatten` предсказуем всегда.



### Задача 7. `newaxis`, `squeeze`, транспонирование.

Создайте `v = np.arange(5)`. Получите:

* `v[:, None]` — форма?
* `v[None, :]` — форма?
* `np.squeeze(v[:, None])` — форма?
* `np.arange(6).reshape(2, 3).T` — форма и содержимое.


In [88]:
v = np.arange(5)
v1 = v[:, None]
print('v[:, None].shape', v1.shape)
print('v[:, None].base is v', v1.base is v )
v2 = v[None, :]
print('v[None, :].shape', v2.shape)
print('v[None, :].base is v', v2.base is v )
v3 = np.squeeze(v[:, None])
print('np.squeeze(v[:, None]).shape', v3.shape)
print('np.squeeze(v[:, None]).base is v', v3.base is v )
v4 = np.arange(6).reshape(2, 3).T
print('np.arange(6).reshape(2, 3).T.shape', v4.shape)
print('np.arange(6).reshape(2, 3).T.base', v4.base)
print('np.arange(6).reshape(2, 3).T.flags.c_contiguous', v4.flags.c_contiguous)
print(v4)


v[:, None].shape (5, 1)
v[:, None].base is v True
v[None, :].shape (1, 5)
v[None, :].base is v True
np.squeeze(v[:, None]).shape (5,)
np.squeeze(v[:, None]).base is v True
np.arange(6).reshape(2, 3).T.shape (3, 2)
np.arange(6).reshape(2, 3).T.base [0 1 2 3 4 5]
np.arange(6).reshape(2, 3).T.flags.c_contiguous False
[[0 3]
 [1 4]
 [2 5]]


Для каждой операции объясните одной строкой, что произошло. Затем вычислите `v[:, None] + v[None, :]` — какая форма получилась и что это за матрица (подсказка: таблица умножения).

> Ответ

* `v[:, None]` → `(5, 1)`: добавлена новая ось длины 1 после первой, одномерный массив стал столбцом (view, данные не копируются).
* `v[None, :]` → `(1, 5)`: новая ось в начале, массив стал строкой (view).
* `np.squeeze(v[:, None])` → `(5,)`: удалены все оси длины 1, вернулись к исходной форме (view).
* `np.arange(6).reshape(2, 3).T` → `(3, 2)`: строки и столбцы поменялись местами, `[[0, 3], [1, 4], [2, 5]]`; это view с переставленными `strides`, поэтому `c_contiguous` стал `False`.

Сумма `v[:, None] + v[None, :]` по правилам broadcasting растягивает `(5, 1)` и `(1, 5)` до `(5, 5)`:

In [89]:
print((v[:, None] + v[None, :]).shape)
print(v[:, None] + v[None, :])
print('таблица умножения:')
print(v[:, None] * v[None, :])

(5, 5)
[[0 1 2 3 4]
 [1 2 3 4 5]
 [2 3 4 5 6]
 [3 4 5 6 7]
 [4 5 6 7 8]]
таблица умножения:
[[ 0  0  0  0  0]
 [ 0  1  2  3  4]
 [ 0  2  4  6  8]
 [ 0  3  6  9 12]
 [ 0  4  8 12 16]]


Форма `(5, 5)`, элемент `[i, j] = i + j`, то есть это **таблица сложения**. Подсказка про таблицу умножения относится к варианту с `*` (он показан выше): `[i, j] = i * j`.



### Задача 8. Транспонирование без копии.

Создайте `A = np.arange(6).reshape(2, 3)`. Получите `B = A.T`. Проверьте `B.base is A` (или `B.base is not None`). Измените `B[0, 1] = 999` — изменился ли `A`? 

> Надо проверять A.base (потому что A само получено из другого массива)

In [90]:
a = np.arange(6).reshape(2, 3)
b = a.T

print('B.base is A.base: ', b.base is a.base)

print('До изменения B[0, 1] = 999')
print(f'{a[1,0]=}')
b[0, 1] = 999
print('После изменения B[0, 1] = 999')
print(f'{a[1,0]=}')
print('A.strides: ', a.strides)
print('B.strides: ', b.strides)



B.base is A.base:  True
До изменения B[0, 1] = 999
a[1,0]=np.int64(3)
После изменения B[0, 1] = 999
a[1,0]=np.int64(999)
A.strides:  (24, 8)
B.strides:  (8, 24)


Сделайте вывод: почему `.T` — view, а не копия, и как это связано с `strides`.

> Вывод

Транспонирование не требует перемещать данные: достаточно поменять местами `shape` и `strides`. У `A` `strides = (24, 8)` (шаг на следующую строку — 24 байта, на следующий столбец — 8), у `B = A.T` — `(8, 24)`. Память одна и та же, поэтому `B[0, 1] = 999` изменило `A[1, 0]` (было 3, стало 999), и операция стоит O(1) независимо от размера. Цена: `B` не `C_CONTIGUOUS`, а последующие операции, требующие непрерывной памяти (`reshape`, `ravel`), могут скопировать данные.

## Блок 3. Операции, агрегации, `axis` (задачи 9–11)

### Задача 9. Поэлементно vs матричное.

Создайте `A = np.arange(6).reshape(2, 3)` и `B = np.arange(6).reshape(3, 2)`. Выполните `A @ B`, `A.T @ B.T`, `A * A`. Выведите формы и значения. 

In [91]:
a = np.arange(6).reshape(2, 3)
b = np.arange(6).reshape(3, 2)
ab = a @ b
atbt = a.T @ b.T
aa = a * a
print(f'A         shape {a.shape}')
print(f'B         shape {b.shape}')
print(f'A @ B     shape {ab.shape}\n{ab}')
print(f'A.T @ B.T shape {atbt.shape}\n{atbt}')
print(f'A * A     shape {aa.shape}\n{aa}')


A         shape (2, 3)
B         shape (3, 2)
A @ B     shape (2, 2)
[[10 13]
 [28 40]]
A.T @ B.T shape (3, 3)
[[ 3  9 15]
 [ 4 14 24]
 [ 5 19 33]]
A * A     shape (2, 3)
[[ 0  1  4]
 [ 9 16 25]]


Объясните: почему `A * B` даёт ошибку (или работает?) и в чём разница между `*` и `@`. Приведите пример, когда `*` и `@` дают **одинаковый** результат (подсказка: диагональная матрица).

> Ответ


In [92]:
try:
    a * b
except ValueError as e:
    print('ValueError:', e)

D1 = np.diag([1, 2, 3])
D2 = np.diag([4, 5, 6])
print(D1 @ D2)
print(D1 * D2)
print('одинаковые:', np.array_equal(D1 @ D2, D1 * D2))

ValueError: operands could not be broadcast together with shapes (2,3) (3,2) 
[[ 4  0  0]
 [ 0 10  0]
 [ 0  0 18]]
[[ 4  0  0]
 [ 0 10  0]
 [ 0  0 18]]
одинаковые: True



* `A * B` даёт ошибку: `*` — поэлементное умножение, формы должны совпадать или быть совместимыми для broadcasting, а `(2, 3)` и `(3, 2)` несовместимы. `A * A` работает, потому что формы одинаковые.
* `@` — матричное умножение (строка на столбец): внутренние размерности должны совпасть, `(2, 3) @ (3, 2) → (2, 2)`, а `A.T @ B.T` — `(3, 2) @ (2, 3) → (3, 3)`.
* Для двух диагональных матриц обе операции дают одно и то же: во внеугольных позициях поэлементное произведение равно 0·0 = 0, а в матричном произведении `(D1 @ D2)[i, j] = Σ D1[i, k]·D2[k, j]` ненулевым остаётся только член k = i = j, то есть произведение диагональных элементов.


### Задача 10. `axis=0` vs `axis=1`.
Создайте `M = np.arange(12).reshape(3, 4)`. Вычислите `M.sum(axis=0)`, `M.sum(axis=1)`, `M.mean(axis=0)`, `M.mean(axis=1)`, `M.argmax(axis=0)`, `M.argmax(axis=1)`. 

In [93]:
M = np.arange(12).reshape(3, 4)
print(f'{M.sum(axis=0)    = }')
print(f'{M.sum(axis=1)    = }')
print(f'{M.mean(axis=0)   = }')
print(f'{M.mean(axis=1)   = }')
print(f'{M.argmax(axis=0) = }')
print(f'{M.argmax(axis=1) = }')


M.sum(axis=0)    = array([12, 15, 18, 21])
M.sum(axis=1)    = array([ 6, 22, 38])
M.mean(axis=0)   = array([4., 5., 6., 7.])
M.mean(axis=1)   = array([1.5, 5.5, 9.5])
M.argmax(axis=0) = array([2, 2, 2, 2])
M.argmax(axis=1) = array([3, 3, 3])


Для каждой операции объясните одной строкой: по какой оси «свернули» и что означает результат. Проверьте на маленьком примере, что `M.sum(axis=0)` — это сумма по столбцам, а не по строкам.

> Ответ

`M` имеет форму `(3, 4)`: ось 0 — строки (3 шт.), ось 1 — столбцы (4 шт.). Параметр `axis` указывает, какая ось **исчезает** (сворачивается).

* `M.sum(axis=0)` = `[12, 15, 18, 21]` — свернули строки (ось 0), получили сумму каждого из 4 столбцов; форма `(4,)`.
* `M.sum(axis=1)` = `[6, 22, 38]` — свернули столбцы (ось 1), получили сумму каждой из 3 строк; форма `(3,)`.
* `M.mean(axis=0)` = `[4, 5, 6, 7]` — среднее по каждому столбцу.
* `M.mean(axis=1)` = `[1.5, 5.5, 9.5]` — среднее по каждой строке.
* `M.argmax(axis=0)` = `[2, 2, 2, 2]` — для каждого столбца номер **строки** с максимумом (везде последняя строка, т. к. значения растут вниз).
* `M.argmax(axis=1)` = `[3, 3, 3]` — для каждой строки номер **столбца** с максимумом (везде последний столбец).

Проверка на маленьком примере:

In [94]:
s = np.array([[1, 2, 3],
              [4, 5, 6]])
print('sum(axis=0):', s.sum(axis=0), '= 1+4, 2+5, 3+6 (по столбцам)')
print('sum(axis=1):', s.sum(axis=1), '= 1+2+3, 4+5+6 (по строкам)')

sum(axis=0): [5 7 9] = 1+4, 2+5, 3+6 (по столбцам)
sum(axis=1): [ 6 15] = 1+2+3, 4+5+6 (по строкам)



Итак, `axis=0` суммирует элементы, стоящие друг под другом (то есть складывает по столбцам), а результат имеет длину, равную числу столбцов.


### Задача 11. `keepdims` и broadcasting.
Создайте `X = np.random.default_rng(42).normal(size=(5, 3))`. Вычислите `X.mean(axis=0)` (форма `(3,)`) и `X.mean(axis=0, keepdims=True)` (форма `(1, 3)`). Выполните `X - X.mean(axis=0)` и `X - X.mean(axis=0, keepdims=True)` — обе формы должны дать `(5, 3)`. Затем посчитайте `X.std(axis=1)` без `keepdims` и с ним, и нормируйте строки: `(X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)`. Убедитесь, что средние по строкам ≈ 0, std по строкам ≈ 1. 

In [95]:
X = np.random.default_rng(42).normal(size=(5, 3))
xm = X.mean(axis=0)
xm1 = X.mean(axis=0, keepdims=True)
print('X.mean(axis=0)                shape', xm.shape)
print('X.mean(axis=0, keepdims=True) shape', xm1.shape)
print('X.mean(axis=0) совппдает по значениям с X.mean(axis=0, keepdims=True):', np.allclose(xm, xm1))

xx1 = X - xm
xx2 = X - xm1
print('X - X.mean(axis=0)                shape', xx1.shape)
print('X - X.mean(axis=0, keepdims=True) shape', xx2.shape)
print('X - X.mean(axis=0) совпадает с X - X.mean(axis=0, keepdims=True):', np.allclose(xx1, xx2))

xs = X.std(axis=1)
xs1 = X.std(axis=1, keepdims=True)
print('X.std(axis=1)                shape', xs.shape)
print('X.std(axis=1, keepdims=True) shape', xs1.shape)

Y = (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)
with np.printoptions(precision=3):
    print('средние по строкам у нормированного X', Y.mean(axis=1))
    print('std по строкам у нормированного X', Y.std(axis=1))





X.mean(axis=0)                shape (3,)
X.mean(axis=0, keepdims=True) shape (1, 3)
X.mean(axis=0) совппдает по значениям с X.mean(axis=0, keepdims=True): True
X - X.mean(axis=0)                shape (5, 3)
X - X.mean(axis=0, keepdims=True) shape (5, 3)
X - X.mean(axis=0) совпадает с X - X.mean(axis=0, keepdims=True): True
X.std(axis=1)                shape (5,)
X.std(axis=1, keepdims=True) shape (5, 1)
средние по строкам у нормированного X [ 3.701e-17 -7.401e-17 -1.850e-17 -3.701e-17 -1.203e-16]
std по строкам у нормированного X [1. 1. 1. 1. 1.]


Объясните, зачем нужен `keepdims`.

> Ответ

`keepdims=True` оставляет свёрнутую ось в результате с длиной 1: вместо `(5,)` получается `(5, 1)`, вместо `(3,)` — `(1, 3)`. Такая форма корректно растягивается (broadcasting) против исходного массива без ручного `reshape`.
Это важно для статистик по строкам (`axis=1`): `X.std(axis=1)` имеет форму `(5,)`, а broadcasting выравнивает формы по **последней** оси, то есть `(5,)` пытается встать против 3 столбцов:



In [96]:
try:
    X - X.std(axis=1)
except ValueError as e:
    print('ValueError:', e)

Z = np.arange(9.).reshape(3, 3)
print('квадратная матрица, без keepdims:', (Z - Z.mean(axis=1)).sum(axis=1))
print('квадратная матрица, с keepdims   :', (Z - Z.mean(axis=1, keepdims=True)).sum(axis=1))

ValueError: operands could not be broadcast together with shapes (5,3) (5,) 
квадратная матрица, без keepdims: [-9.  0.  9.]
квадратная матрица, с keepdims   : [0. 0. 0.]


Для прямоугольной матрицы NumPy выдаёт ошибку, а для квадратной (n, n) ошибки нет, и результат **молча неверный** (вычитаются средние по столбцам вместо строк). `keepdims=True` защищает от обеих проблем. Для `axis=0` без `keepdims` форма `(3,)` совпадает с последней осью, поэтому работает, но единый подход с `keepdims=True` безопаснее.


## Блок 4. Индексация, выборки, broadcasting (задачи 12–15)

### Задача 12. Срез — view, не копия.
Создайте `a = np.arange(10)`. Получите `b = a[2:5]`. Измените `b[0] = 999` и проверьте `a`. Затем повторите с `c = a[2:5].copy()` — изменился ли `a`? 

In [97]:
a = np.arange(10)
print('a[2] до b[0] = 999:   ', a[2])
b = a[2:5]
b[0] = 999
print('a[2] после b[0] = 999:', a[2])

a = np.arange(10)
print('a[2] до c[0] = 999:   ', a[2])
c = a[2:5].copy()
c[0] = 999
print('a[2] после c[0] = 999:', a[2])


a[2] до b[0] = 999:    2
a[2] после b[0] = 999: 999
a[2] до c[0] = 999:    2
a[2] после c[0] = 999: 2


Объясните: почему это важно для численных методов (подсказка: если вы «обрезаете» данные для анализа и случайно меняете оригинал, что произойдёт с экспериментом?). Приведите пример, когда view — преимущество (экономия памяти), а когда — баг.

> Ответ

Срез `a[2:5]` — это view на ту же память, поэтому `b[0] = 999` изменило `a[2]` (с 2 на 999), а `.copy()` создал независимый массив, и `a` не пострадал. В численных экспериментах это важно, потому что исходные данные должны оставаться неизменными: если «вырезанную» часть чистить, нормировать или обнулять и при этом случайно портится оригинал, то все последующие расчёты идут по искажённым данным, повторный запуск ячейки даёт другой результат, а ошибку трудно найти, ведь формально исходный массив никто не менял.

* View как преимущество: окна и подвыборки больших массивов (например, `data[:1000]` у массива в гигабайты, скользящее окно, столбец `X[:, 0]`) обрабатываются без копирования, то есть быстро и без расхода памяти.
* View как баг: `part = data[:100]; part[part < 0] = 0` или `col = X[:, 0]; col -= col.mean()` молча меняют `data` и `X`. Обратная ситуация: маленький срез от огромного массива удерживает в памяти весь массив; тогда лучше сделать `.copy()`.


### Задача 13. Булева маска и fancy indexing.
Создайте `a = np.random.default_rng(0).normal(size=20)`. Выполните:

* `a[a > 0]` — сколько элементов?
* `a[(a > -1) & (a < 1)]` — сколько?
* `a[[0, 5, 10, 15]]` — что это за выборка?
* `a[np.abs(a) > 2]` — выбросы.

Затем занулите все отрицательные через `a[a < 0] = 0` и выведите результат. 

In [98]:
a = np.random.default_rng(0).normal(size=20)
print('Исходный массив')
with np.printoptions(precision=3):
    print(a)
print(f'a[a > 0] элементов: {len(a[a > 0])}')
print(f'a[(a > -1) & (a < 1)] элементов: {len(a[(a > -1) & (a < 1)])}')
print(f'выборка a[[0, 5, 10, 15]]: {a[[0, 5, 10, 15]]}')
print(f'выбросы a[np.abs(a) > 2]: {a[np.abs(a) > 2]}')
a[a < 0] = 0
print('Зануление всех отрицательных a[a < 0] = 0')
with np.printoptions(precision=3):
    print(a)



Исходный массив
[ 0.126 -0.132  0.64   0.105 -0.536  0.362  1.304  0.947 -0.704 -1.265
 -0.623  0.041 -2.325 -0.219 -1.246 -0.732 -0.544 -0.316  0.412  1.043]
a[a > 0] элементов: 9
a[(a > -1) & (a < 1)] элементов: 15
выборка a[[0, 5, 10, 15]]: [ 0.12573022  0.36159505 -0.62327446 -0.73226735]
выбросы a[np.abs(a) > 2]: [-2.32503077]
Зануление всех отрицательных a[a < 0] = 0
[0.126 0.    0.64  0.105 0.    0.362 1.304 0.947 0.    0.    0.    0.041
 0.    0.    0.    0.    0.    0.    0.412 1.043]


Объясните: почему `a[a > 0] & (a < 1)` даёт ошибку, а `(a > 0) & (a < 1)` — работает.

> Ответ

Ответы на вопросы задачи по результатам вывода: `a[a > 0]` — 9 элементов (положительные), `a[(a > -1) & (a < 1)]` — 15 элементов, `a[[0, 5, 10, 15]]` — fancy indexing, выборка элементов с индексами 0, 5, 10, 15 (результат — копия, а не view), `a[np.abs(a) > 2]` — один выброс (-2.325).


In [99]:
try:
    a[a > 0] & (a < 1)
except TypeError as e:
    print('TypeError:', e)

TypeError: ufunc 'bitwise_and' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''


* `a[a > 0]` — это уже не маска, а **выборка значений**: массив `float64` из 9 элементов. Побитовое `&` для чисел с плавающей точкой не определено (`TypeError`), к тому же длина 9 не совпала бы с длиной маски `(a < 1)`, равной 20.
* `(a > 0)` и `(a < 1)` — булевы массивы одинаковой длины 20; `&` для них — поэлементное логическое И, и результат — маска, которую можно подставить в `a[...]`.
* Скобки обязательны: у `&` приоритет выше, чем у `>` и `<`, поэтому `a > 0 & a < 1` разбирается неправильно. Вместо `and`/`or` для массивов нужно использовать `&`/`|`.



### Задача 14. Broadcasting в реальной задаче.
Создайте `X = np.random.default_rng(1).normal(size=(100, 3))` (100 наблюдений, 3 признака). Нормируйте каждый столбец: вычтите среднее и разделите на std **через broadcasting** (без цикла). Убедитесь, что `X_norm.mean(axis=0)` ≈ 0 и `X_norm.std(axis=0)` ≈ 1. Затем посчитайте матрицу попарных евклидовых расстояний между строками: `D[i, j] = ||X_norm[i] - X_norm[j]||` — через broadcasting в одну строку, без цикла. Проверьте, что `D.shape = (100, 100)`, диагональ ≈ 0, матрица симметрична.


In [100]:
X = np.random.default_rng(1).normal(size=(100, 3))
X_norm = (X - X.mean(axis=0, keepdims=True)) / X.std(axis=0, keepdims=True)
print('X_norm.mean(axis=0) ≈ 0:', np.allclose(X_norm.mean(axis=0), 0))
print('X_norm.std(axis=0) ≈ 1 :', np.allclose(X_norm.std(axis=0), 1))
D = np.linalg.norm(X_norm[:, None, :] - X_norm[None, :, :], axis=-1)
print('D.shape                :', D.shape)
print('диагональ ≈ 0          :', np.allclose(np.diag(D), 0))
print('матрица симметрична    :', np.allclose(D, D.T))


X_norm.mean(axis=0) ≈ 0: True
X_norm.std(axis=0) ≈ 1 : True
D.shape                : (100, 100)
диагональ ≈ 0          : True
матрица симметрична    : True



**Задача 15. Полный кейс: загрузка → выборка → reshape → агрегация → нормировка.**
Сгенерируйте «измерения»: `rng = np.random.default_rng(42)`, 30 дней × 4 датчика, нормальное распределение со средним 20 и std 2, плюс один искусственный выброс (`raw[7, 2] = 100`).

Выполните по шагам и выведите результат каждого:

1. `shape`, `dtype`, `size`.
2. Найдите выбросы булевой маской: `np.abs(raw - raw.mean(axis=0)) > 3 * raw.std(axis=0)`. Сколько их?
3. Замените выбросы на `np.nan`, затем на среднее по столбцу через `np.nanmean` и `np.where`.
4. Преобразуйте `(30, 4)` в `(4, 30)` через `.T`, затем обратно через `.T` — убедитесь через `np.array_equal`, что данные не изменились.
5. «Сплющите» в `(120,)` через `ravel`, затем восстановите `(30, 4)` через `reshape` — проверьте равенство.
6. Посчитайте средние и std по каждому датчику (`axis=0`).
7. Нормируйте каждый столбец через broadcasting. Убедитесь, что средние ≈ 0, std ≈ 1.


In [101]:
rng = np.random.default_rng(42)
raw = rng.normal(loc=20, scale=2, size=(30, 4))
raw[7, 2] = 100

print('1) Информация о массиве')
print('shape: ', raw.shape)
print('dtype: ', raw.dtype)
print('size: ', raw.size)

print('2) Выбросы > 3 sigma')
out = np.abs(raw - raw.mean(axis=0)) > 3 * raw.std(axis=0)
print('Количество выбросов: ', out.sum())

print('3) замена выбросов на среднее')
raw[out] = np.nan
raw = np.where(np.isnan(raw), np.nanmean(raw, axis=0), raw)

print('4) Преобразование через .T')
raw_t = raw.T
print('raw.T shape:', raw_t.shape)
raw_t_t = raw_t.T
print('raw.T.T shape:', raw_t_t.shape)
print('raw.T.T == raw:', np.array_equal(raw, raw_t_t))

print('5) ravel и reshape')
flat = raw.ravel()
flat_r = flat.reshape((30, 4))
print('raw.raven().reshape(30,4) is raw:',flat_r.base is raw)

print('6) средние и std по каждому датчику')
mean = raw.mean(axis=0)
std = raw.std(axis=0)
print('7) нормировка')
norm = (raw - mean) / std
print('norm.mean(axis=0) ≈ 0:', np.allclose(norm.mean(axis=0), 0))
print('norm.std(axis=0) ≈ 1 :', np.allclose(norm.std(axis=0), 1))


1) Информация о массиве
shape:  (30, 4)
dtype:  float64
size:  120
2) Выбросы > 3 sigma
Количество выбросов:  1
3) замена выбросов на среднее
4) Преобразование через .T
raw.T shape: (4, 30)
raw.T.T shape: (30, 4)
raw.T.T == raw: True
5) ravel и reshape
raw.raven().reshape(30,4) is raw: True
6) средние и std по каждому датчику
7) нормировка
norm.mean(axis=0) ≈ 0: True
norm.std(axis=0) ≈ 1 : True


### Вывод

Копия создаётся в `np.where(...)`, который возвращает новый массив (после шага 3 `raw` — уже другой объект, а если исходные данные с выбросом нужны, то заранее надо сделать `raw.copy()`); присваивание `raw[out] = np.nan` до этого меняет массив на месте, а булева маска при чтении (`raw[out]`) тоже дала бы копию. `.T`, `ravel()` (массив непрерывен) и `reshape` дали view без копирования. Broadcasting сработал на шагах 2, 3 и 7, где вектор статистик формы `(4,)` растягивается на все 30 строк `(30, 4)` (`raw - raw.mean(axis=0)`, `np.nanmean(raw, axis=0)`, `(raw - mean) / std`). Ошибка формы возможна при нормировке по неверной оси (статистики по `axis=1` формы `(30,)` не выровняются с `(30, 4)` без `keepdims=True`) и при подмене `.T` на `reshape(4, 30)`: форма совпадёт, но данные будут перемешаны, и `np.array_equal` это поймает.

